# Music Annotation Project
## Notebook 72: Frozen V8.8 Annotated-Only Supervised Operational Trial

**Runtime: GPU recommended.** Select a T4 GPU for the unified V8.8 `p_print` stage, then **Runtime > Run all**.

This amended Notebook 72 processes every eligible reviewed page from `source/annotated_only` and `source/annotated` through one identical clean-free runtime path. It uses the unified `v8_8_operator.py` with two content-addressed learned bundles:

1. V8.2F full-development calibrated `p_print` provider;
2. 52F full-development calibrated SAFE_REMOVE provider.

### Non-negotiable contract
- reviewer-corrected authoritative masks only;
- no silent fallback to unreviewed YOLO masks;
- no clean page, truth label, clean-derived map or outcome may be read before the prediction lock closes;
- no retraining, recalibration, schema change or threshold movement;
- runtime outputs, decisions, probability maps and corrected pages are locked before clean truth becomes available;
- outside the prepared authoritative mask is byte-for-byte immutable;
- unsupported or invalid cases fail closed;
- post-lock clean truth is evaluation only;
- no automatic production promotion.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow joblib scikit-learn torch torchvision matplotlib pillow

## 1. Configuration and immutable run directory

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import asdict
import builtins, hashlib, importlib.util, json, os, platform, shutil, sys, time, zipfile
import cv2, joblib, numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
SOURCE_DIRS=[ROOT/'source'/'annotated_only', ROOT/'source'/'annotated']
REVIEW_AUDIT_DIR=ROOT/'human_in_the_middle'/'audit'
DRAFT_AUDIT_DIR=ROOT/'human_in_the_middle'/'draft_audit'

# Pin these deliberately for the definitive trial if desired.
OPERATOR_PATH_OVERRIDE=ROOT/'v8_8_operator.py'
P_PRINT_BUNDLE_ROOT_OVERRIDE=None
REMOVE_BUNDLE_ROOT_OVERRIDE=None

P_PRINT_FREEZE_ROOT=ROOT/'observed_print_full_development_freeze_v8_2f'
REMOVE_FREEZE_ROOT=ROOT/'m5_remove_full_development_freeze_v8_8f'
OUTPUT_ROOT=ROOT/'notebook72_frozen_v8_8_annotated_only_operational_trial'
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'

SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
RUN_ID=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook72_unified_v88'
OUT=OUTPUT_ROOT/RUN_ID
INPUTS=OUT/'inputs'; PAGES=OUT/'pages'; TABLES=OUT/'tables'; AUDIT=OUT/'audit'; EVAL=OUT/'post_lock_evaluation'; REVIEW=OUT/'blinded_review'; UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,PAGES,TABLES,AUDIT,EVAL,REVIEW,UPLOAD]: p.mkdir(parents=True,exist_ok=False)

LOCK_STATE='OPEN'
CLEAN_READS_BEFORE_LOCK=[]
ACCESS_ROWS=[]
print('Device:',DEVICE)
print('Output:',OUT)

## 2. Helpers, semantic hashes and truth-firewall instrumentation

In [ ]:
def require(ok,msg):
    if not bool(ok): raise RuntimeError(msg)
def sha256_file(path):
    path=Path(path); require(path.is_file(),f'Missing file: {path}')
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()
def object_hash(x): return hashlib.sha256(json.dumps(x,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def atomic_json(path,value):
    path=Path(path); tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str)); tmp.replace(path)
def image_map(folder): return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if Path(folder).exists() else {}
def latest_complete(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

def guarded_read_image(path,flags=cv2.IMREAD_GRAYSCALE,purpose='runtime'):
    path=Path(path).resolve(); is_clean=(ROOT/'source'/'clean').resolve() in path.parents
    ACCESS_ROWS.append({'utc':datetime.now(timezone.utc).isoformat(),'path':str(path),'purpose':purpose,'lock_state':LOCK_STATE,'clean':is_clean})
    if LOCK_STATE!='CLOSED' and is_clean:
        CLEAN_READS_BEFORE_LOCK.append(str(path)); raise RuntimeError(f'TRUTH FIREWALL: clean read before prediction lock: {path}')
    im=cv2.imread(str(path),flags); require(im is not None,f'Unreadable image: {path}'); return im

def md5_file(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

## 3. Resolve and verify the unified V8.8 operator and both frozen bundles

In [ ]:
OPERATOR_PATH=Path(OPERATOR_PATH_OVERRIDE)
require(OPERATOR_PATH.is_file(),f'Missing unified operator: {OPERATOR_PATH}')
P_PRINT_BUNDLE=Path(P_PRINT_BUNDLE_ROOT_OVERRIDE) if P_PRINT_BUNDLE_ROOT_OVERRIDE else latest_complete(P_PRINT_FREEZE_ROOT,['bundle_manifest.json','p_print_model.pt','p_print_calibrator.joblib','provider_policy.json','channel_schema.json','p_print_provider.py','replay/replay_test.json'])
REMOVE_BUNDLE=Path(REMOVE_BUNDLE_ROOT_OVERRIDE) if REMOVE_BUNDLE_ROOT_OVERRIDE else latest_complete(REMOVE_FREEZE_ROOT,['bundle_manifest.json','remove_estimator.joblib','remove_calibrator.joblib','policy.json','predictor_schema.json','replay/replay_test.json'])
require(P_PRINT_BUNDLE is not None,'No complete V8.2F p_print bundle found.')
require(REMOVE_BUNDLE is not None,'No complete 52F REMOVE bundle found.')

pman=json.loads((P_PRINT_BUNDLE/'bundle_manifest.json').read_text()); rman=json.loads((REMOVE_BUNDLE/'bundle_manifest.json').read_text())
require(pman.get('unseen_page_inference') is True,'p_print bundle does not authorise unseen-page inference.')
require(rman.get('unseen_page_scoring_available') is True,'REMOVE bundle does not authorise unseen-page scoring.')
require(pman.get('clean_truth_required_at_inference') is False,'p_print bundle requires truth.')

spec=importlib.util.spec_from_file_location('v8_8_operator',OPERATOR_PATH); require(spec and spec.loader,'Cannot load v8_8_operator.py')
v88=importlib.util.module_from_spec(spec); sys.modules['v8_8_operator']=v88; spec.loader.exec_module(v88)
operator=v88.load_operator(P_PRINT_BUNDLE,REMOVE_BUNDLE,device=DEVICE)
capabilities=asdict(operator.capabilities)
require(capabilities.get('unseen_page_inference') is True,'Unified operator does not support unseen-page inference.')
require(capabilities.get('requires_external_decisions') is False,'Unified operator unexpectedly requires historical external decisions.')
require(capabilities.get('uses_clean_truth') is False,'Unified operator is not truth-free.')

operator_identity={'operator_path':str(OPERATOR_PATH),'operator_sha256':sha256_file(OPERATOR_PATH),'operator_bundle_id':operator.bundle_id,'capabilities':capabilities,'p_print_bundle':str(P_PRINT_BUNDLE),'p_print_bundle_id':pman['bundle_id'],'p_print_manifest_sha256':sha256_file(P_PRINT_BUNDLE/'bundle_manifest.json'),'remove_bundle':str(REMOVE_BUNDLE),'remove_bundle_id':rman['bundle_id'],'remove_manifest_sha256':sha256_file(REMOVE_BUNDLE/'bundle_manifest.json'),'device':DEVICE}
atomic_json(INPUTS/'operator_identity.json',operator_identity)
print(json.dumps(operator_identity,indent=2))

## 4. Build the complete reviewed runtime roster with auditable HITL lineage

In [ ]:
def read_audits(folder,label):
    records=[]; failures=[]
    for p in sorted(Path(folder).glob('*.json')):
        try:
            x=json.loads(p.read_text()); require(isinstance(x,dict),f'Non-object audit: {p}'); x['_audit_path']=str(p); x['_kind']=label; records.append(x)
        except Exception as e: failures.append({'path':str(p),'kind':label,'error':str(e)})
    return records,failures
reviews,rf=read_audits(REVIEW_AUDIT_DIR,'review'); drafts,dfail=read_audits(DRAFT_AUDIT_DIR,'draft')
if rf or dfail:
    pd.DataFrame(rf+dfail).to_csv(AUDIT/'hitl_audit_parse_failures.csv',index=False); raise RuntimeError('Unreadable HITL audit records; see audit table.')

def matches(records,source_md5): return [x for x in records if str(x.get('source_md5','')).lower()==source_md5.lower()]
def resolve_review(source,population):
    smd5=md5_file(source); rr=matches(reviews,smd5); dd=matches(drafts,smd5)
    if len(rr)>1: return None,'DUPLICATE_REVIEW_AUDIT'
    if not rr: return None,'NO_REVIEWED_AUTHORITATIVE_MASK'
    r=rr[0]; observed=Path(r.get('aligned_output','')); mask=Path(r.get('corrected_mask_output',''))
    if not observed.is_file(): return None,'REVIEW_ALIGNED_OUTPUT_MISSING'
    if not mask.is_file(): return None,'REVIEW_CORRECTED_MASK_MISSING'
    return {'page':source.stem,'population':population,'source_path':str(source),'source_md5':smd5,'source_sha256':sha256_file(source),'observed_path':str(observed),'observed_sha256':sha256_file(observed),'seed_mask_path':str(mask),'seed_mask_sha256':sha256_file(mask),'review_audit_path':r['_audit_path'],'review_audit_sha256':sha256_file(r['_audit_path']),'draft_audit_path':dd[0]['_audit_path'] if len(dd)==1 else None},None

rows=[]; exclusions=[]
for folder in SOURCE_DIRS:
    population=folder.name
    for source in sorted(image_map(folder).values()):
        rec,reason=resolve_review(source,population)
        if reason: exclusions.append({'page':source.stem,'population':population,'source_path':str(source),'reason':reason})
        else: rows.append(rec)
roster=pd.DataFrame(rows); excluded=pd.DataFrame(exclusions)
require(len(roster)>0,'No eligible reviewed pages.')
require(not roster[['population','page']].duplicated().any(),'Duplicate population/page identities.')
# Content duplicates are audit-visible and excluded only when exact source bytes duplicate another roster row.
roster['content_duplicate']=roster.source_sha256.duplicated(keep=False)
roster.to_csv(INPUTS/'runtime_roster.csv',index=False); excluded.to_csv(INPUTS/'runtime_exclusions.csv',index=False)
print('Eligible reviewed pages:',len(roster)); print('Excluded:',len(excluded)); print(roster.groupby('population').size())

## 5. Explicit truth-firewall preflight

In [ ]:
require(LOCK_STATE=='OPEN','Unexpected lock state.')
require(len(CLEAN_READS_BEFORE_LOCK)==0,'Clean data was read before runtime.')
for c in roster.columns:
    require('clean' not in c.lower() and 'truth' not in c.lower() and 'outcome' not in c.lower(),f'Forbidden runtime roster column: {c}')
truth_firewall={'state':'ENFORCED','clean_reads_before_lock':0,'forbidden_runtime_root':'source/clean','runtime_roster_sha256':sha256_file(INPUTS/'runtime_roster.csv'),'operator_identity_sha256':sha256_file(INPUTS/'operator_identity.json')}
atomic_json(AUDIT/'truth_firewall_preflight.json',truth_firewall)
print(json.dumps(truth_firewall,indent=2))

## 6. Unified frozen inference over every eligible reviewed page

In [ ]:
page_rows=[]; output_hash_rows=[]; failures=[]
for _,r in roster.iterrows():
    page=str(r.page); key=f'{r.population}__{page}'; d=PAGES/key; d.mkdir(parents=True,exist_ok=False)
    try:
        observed=guarded_read_image(r.observed_path,purpose='runtime_observed')
        seed=guarded_read_image(r.seed_mask_path,purpose='runtime_authoritative_seed_mask')
        require(observed.shape==seed.shape,f'{key}: observed/seed shape mismatch')
        t0=time.time(); result=v88.run_page(observed=observed,authoritative_seed_mask=seed,operator=operator,page=key); seconds=time.time()-t0
        corrected=np.asarray(result.corrected); prepared=np.asarray(result.authoritative_mask,bool); baseline=np.asarray(result.baseline_v78); safe=np.asarray(result.safe_remove_mask,bool); m3=np.asarray(result.m3_mask,bool); m5=np.asarray(result.m5_remove_mask,bool)
        require(corrected.shape==observed.shape and corrected.dtype==observed.dtype,f'{key}: corrected shape/dtype mismatch')
        require(prepared.shape==observed.shape,f'{key}: prepared mask mismatch')
        require(np.array_equal(corrected[~prepared],observed[~prepared]),f'{key}: outside-mask drift')
        changed=corrected!=observed
        cv2.imwrite(str(d/'observed.png'),observed); cv2.imwrite(str(d/'authoritative_seed_mask.png'),np.where(seed>0,255,0).astype(np.uint8)); cv2.imwrite(str(d/'prepared_mask.png'),prepared.astype(np.uint8)*255); cv2.imwrite(str(d/'baseline_v78.png'),baseline); cv2.imwrite(str(d/'corrected.png'),corrected); cv2.imwrite(str(d/'safe_remove.png'),safe.astype(np.uint8)*255); cv2.imwrite(str(d/'m3.png'),m3.astype(np.uint8)*255); cv2.imwrite(str(d/'m5_remove.png'),m5.astype(np.uint8)*255); cv2.imwrite(str(d/'reason_map.png'),np.asarray(result.reason_map,np.uint8)); cv2.imwrite(str(d/'changed.png'),changed.astype(np.uint8)*255)
        np.savez_compressed(d/'probabilities.npz',p_print=np.asarray(result.p_print),p_remove=np.asarray(result.p_remove))
        result.features.to_parquet(d/'observable_features.parquet',index=False)
        atomic_json(d/'page_audit.json',result.audit)
        output_hash_rows.append({'page_key':key,'observed_array_sha256':array_hash(observed),'seed_mask_array_sha256':array_hash(seed),'prepared_mask_array_sha256':array_hash(prepared),'baseline_array_sha256':array_hash(baseline),'corrected_array_sha256':array_hash(corrected),'safe_remove_array_sha256':array_hash(safe),'m3_array_sha256':array_hash(m3),'m5_remove_array_sha256':array_hash(m5),'reason_array_sha256':array_hash(result.reason_map),'p_print_array_sha256':array_hash(result.p_print),'p_remove_array_sha256':array_hash(result.p_remove),'features_sha256':sha256_file(d/'observable_features.parquet')})
        page_rows.append({'page_key':key,'page':page,'population':r.population,'shape':str(observed.shape),'prepared_mask_pixels':int(prepared.sum()),'changed_pixels':int(changed.sum()),'safe_remove_pixels':int(safe.sum()),'m3_pixels':int(m3.sum()),'m5_remove_pixels':int(m5.sum()),'outside_mask_drift':int(np.sum(corrected[~prepared]!=observed[~prepared])),'runtime_seconds':seconds,'status':'PASS'})
        print('Processed',key,'SAFE_REMOVE',int(safe.sum()),'changed',int(changed.sum()))
    except Exception as e:
        failures.append({'page_key':key,'page':page,'population':r.population,'error':repr(e)}); print('FAILED',key,e)
if failures:
    pd.DataFrame(failures).to_csv(AUDIT/'runtime_failures.csv',index=False); raise RuntimeError(f'Runtime failures on {len(failures)} page(s); run invalidated.')
pages_df=pd.DataFrame(page_rows); hashes_df=pd.DataFrame(output_hash_rows); pages_df.to_csv(TABLES/'runtime_page_summary.csv',index=False); hashes_df.to_csv(TABLES/'page_output_hashes.csv',index=False)
require((pages_df.outside_mask_drift==0).all(),'Outside-mask drift detected.')

## 7. Determinism replay gate before prediction lock

In [ ]:
# Re-run a fixed deterministic subset: first, middle and last roster pages.
pos=sorted(set([0,len(roster)//2,len(roster)-1])); replay_rows=[]
for i in pos:
    r=roster.iloc[i]; key=f'{r.population}__{r.page}'; d=PAGES/key
    observed=guarded_read_image(r.observed_path,purpose='runtime_replay_observed'); seed=guarded_read_image(r.seed_mask_path,purpose='runtime_replay_seed')
    rr=v88.run_page(observed=observed,authoritative_seed_mask=seed,operator=operator,page=key)
    expected=hashes_df.set_index('page_key').loc[key]
    replay_rows.append({'page_key':key,'corrected_exact':array_hash(rr.corrected)==expected.corrected_array_sha256,'safe_remove_exact':array_hash(rr.safe_remove_mask)==expected.safe_remove_array_sha256,'m5_remove_exact':array_hash(rr.m5_remove_mask)==expected.m5_remove_array_sha256,'p_print_exact':array_hash(rr.p_print)==expected.p_print_array_sha256,'p_remove_exact':array_hash(rr.p_remove)==expected.p_remove_array_sha256})
replay_df=pd.DataFrame(replay_rows); replay_df['passed']=replay_df.drop(columns='page_key').all(axis=1); replay_df.to_csv(AUDIT/'replay_test.csv',index=False)
replay={'pages':len(replay_df),'passed':bool(replay_df.passed.all()),'operator_bundle_id':operator.bundle_id,'replay_csv_sha256':sha256_file(AUDIT/'replay_test.csv')}; atomic_json(AUDIT/'replay_test.json',replay); require(replay['passed'],'Unified operator deterministic replay failed.')
print(json.dumps(replay,indent=2))

## 8. Close prediction/output lock

In [ ]:
pd.DataFrame(ACCESS_ROWS).to_csv(AUDIT/'runtime_access_audit.csv',index=False)
require(len(CLEAN_READS_BEFORE_LOCK)==0,'Truth firewall violation before lock.')
lock_payload={'runtime_roster_sha256':sha256_file(INPUTS/'runtime_roster.csv'),'runtime_exclusions_sha256':sha256_file(INPUTS/'runtime_exclusions.csv'),'operator_identity_sha256':sha256_file(INPUTS/'operator_identity.json'),'operator_bundle_id':operator.bundle_id,'page_output_hashes_sha256':sha256_file(TABLES/'page_output_hashes.csv'),'runtime_page_summary_sha256':sha256_file(TABLES/'runtime_page_summary.csv'),'replay_test_sha256':sha256_file(AUDIT/'replay_test.json'),'runtime_access_audit_sha256':sha256_file(AUDIT/'runtime_access_audit.csv'),'clean_reads_before_lock':0,'software':{'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'torch':torch.__version__,'cuda':torch.version.cuda,'platform':platform.platform()}}
lock_record={'created_utc':datetime.now(timezone.utc).isoformat(),'state':'CLOSED','lock_id':object_hash(lock_payload),'payload':lock_payload}
atomic_json(OUT/'prediction_lock.json',lock_record)
LOCK_STATE='CLOSED'
print('PREDICTION LOCK CLOSED:',lock_record['lock_id'])

## 9. Post-lock paired clean-truth evaluation only where clean counterpart exists

In [ ]:
require(LOCK_STATE=='CLOSED','Post-lock evaluation is forbidden before prediction lock closure.')
# Lock integrity must still match before clean is resolved.
lock= json.loads((OUT/'prediction_lock.json').read_text())
require(lock['payload']['runtime_roster_sha256']==sha256_file(INPUTS/'runtime_roster.csv'),'Roster changed after lock.')
require(lock['payload']['page_output_hashes_sha256']==sha256_file(TABLES/'page_output_hashes.csv'),'Outputs changed after lock.')

CLEAN_DIR=ROOT/'source'/'clean'; clean_map=image_map(CLEAN_DIR)
def printed_ink(im): return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
metrics=[]
for _,r in roster.iterrows():
    key=f'{r.population}__{r.page}'; clean_path=clean_map.get(str(r.page))
    if clean_path is None:
        metrics.append({'page_key':key,'page':r.page,'population':r.population,'paired_clean':False}); continue
    clean=guarded_read_image(clean_path,purpose='post_lock_clean_evaluation'); observed=guarded_read_image(r.observed_path,purpose='post_lock_observed_evaluation'); corrected=guarded_read_image(PAGES/key/'corrected.png',purpose='post_lock_corrected_evaluation')
    if clean.shape!=corrected.shape:
        metrics.append({'page_key':key,'page':r.page,'population':r.population,'paired_clean':False,'evaluation_failure':'shape_mismatch'}); continue
    diff=np.abs(clean.astype(np.float32)-corrected.astype(np.float32)); before=np.abs(clean.astype(np.float32)-observed.astype(np.float32)); ci=printed_ink(clean); oi=printed_ink(observed); xi=printed_ink(corrected)
    metrics.append({'page_key':key,'page':r.page,'population':r.population,'paired_clean':True,'corrected_mae':float(diff.mean()),'observed_mae':float(before.mean()),'mae_improvement':float(before.mean()-diff.mean()),'clean_ink_pixels':int(ci.sum()),'observed_ink_pixels':int(oi.sum()),'corrected_ink_pixels':int(xi.sum()),'corrected_vs_clean_ink_xor':int(np.logical_xor(ci,xi).sum())})
eval_df=pd.DataFrame(metrics); eval_df.to_csv(EVAL/'paired_truth_evaluation.csv',index=False)
print('Paired clean evaluation pages:',int(eval_df.paired_clean.fillna(False).sum()),'of',len(eval_df))

## 10. Blinded whole-page review package for the entire runtime population

In [ ]:
# No clean truth is displayed. Reviewer compares annotated observed vs corrected at identical scale.
def make_pair(left,right,label):
    h=max(left.shape[0],right.shape[0]); gap=20; canvas=np.full((h+45,left.shape[1]+right.shape[1]+gap),255,np.uint8); canvas[45:45+left.shape[0],:left.shape[1]]=left; canvas[45:45+right.shape[0],left.shape[1]+gap:]=right; cv2.putText(canvas,'A',(10,30),cv2.FONT_HERSHEY_SIMPLEX,.8,40,2); cv2.putText(canvas,'B',(left.shape[1]+gap+10,30),cv2.FONT_HERSHEY_SIMPLEX,.8,40,2); return canvas
review_rows=[]
for _,r in roster.iterrows():
    key=f'{r.population}__{r.page}'; observed=guarded_read_image(r.observed_path,purpose='post_lock_blinded_review_observed'); corrected=guarded_read_image(PAGES/key/'corrected.png',purpose='post_lock_blinded_review_corrected'); pair=make_pair(observed,corrected,key); path=REVIEW/f'{key}_A_B.png'; cv2.imwrite(str(path),pair); review_rows.append({'page_key':key,'page':r.page,'population':r.population,'review_image':str(path),'review_image_sha256':sha256_file(path),'preferred':'','visible_annotation_residue':'','visible_print_damage':'','visible_print_recovery':'','no_visible_difference':'','notes':''})
pd.DataFrame(review_rows).to_csv(REVIEW/'blinded_review_queue.csv',index=False)
print('Blinded review queue:',REVIEW/'blinded_review_queue.csv')

## 11. Operational summary and fail-closed verdict

In [ ]:
access=pd.DataFrame(ACCESS_ROWS); access.to_csv(AUDIT/'all_access_audit.csv',index=False)
prelock_clean=int(((access.clean==True)&(access.lock_state!='CLOSED')).sum()) if len(access) else 0
checks={'runtime_roster_nonempty':len(roster)>0,'all_runtime_pages_completed':len(pages_df)==len(roster),'outside_mask_drift_zero':bool((pages_df.outside_mask_drift==0).all()),'deterministic_replay_passed':replay['passed'],'prediction_lock_closed':LOCK_STATE=='CLOSED','clean_reads_before_lock_zero':prelock_clean==0,'unseen_page_inference':capabilities['unseen_page_inference'] is True,'external_decisions_not_required':capabilities['requires_external_decisions'] is False,'runtime_truth_free':capabilities['uses_clean_truth'] is False}
first_failed=next((k for k,v in checks.items() if v is not True),None)
verdict='PASS_NOTEBOOK72_UNIFIED_V88_RUNTIME_COMPLETE_REVIEW_PENDING' if first_failed is None else 'FAIL_NOTEBOOK72_RUNTIME_INVALID'
record={'created_utc':datetime.now(timezone.utc).isoformat(),'verdict':verdict,'first_failed_gate':first_failed,'checks':checks,'prediction_lock_id':lock_record['lock_id'],'operator_bundle_id':operator.bundle_id,'runtime_pages':len(roster),'excluded_pages':len(excluded),'paired_clean_pages':int(eval_df.paired_clean.fillna(False).sum()),'review_queue_pages':len(review_rows),'meaning':'PASS establishes completion and lock integrity of the supervised operational trial runtime. It does not constitute production promotion; blinded review remains a separate human evidence step.'}
atomic_json(OUT/'notebook72_verdict.json',record)
summary=['NOTEBOOK 72 - UNIFIED V8.8 ANNOTATED-ONLY SUPERVISED OPERATIONAL TRIAL','',f'Verdict: {verdict}',f'Prediction lock: {lock_record["lock_id"]}',f'Unified operator bundle: {operator.bundle_id}',f'Runtime pages: {len(roster)}',f'Excluded pages: {len(excluded)}',f'Paired clean evaluation pages: {int(eval_df.paired_clean.fillna(False).sum())}',f'Blinded review queue pages: {len(review_rows)}',f'Clean reads before lock: {prelock_clean}',f'Outside-mask drift: {int(pages_df.outside_mask_drift.sum())}','',*([f'{k}: {v}' for k,v in checks.items()]),'','PASS is operational-trial runtime evidence only. No automatic production promotion is granted.']
(OUT/'executive_summary.txt').write_text('\n'.join(summary)); print('\n'.join(summary))
if first_failed: raise RuntimeError(f'Notebook 72 failed at {first_failed}.')

## 12. Standard final upload package

In [ ]:
# Refresh audit after all post-lock accesses.
pd.DataFrame(ACCESS_ROWS).to_csv(AUDIT/'all_access_audit.csv',index=False)
UPLOAD_ARTEFACTS=[OUT/'executive_summary.txt',OUT/'notebook72_verdict.json',OUT/'prediction_lock.json',INPUTS/'runtime_roster.csv',INPUTS/'runtime_exclusions.csv',INPUTS/'operator_identity.json',TABLES/'runtime_page_summary.csv',TABLES/'page_output_hashes.csv',AUDIT/'replay_test.json',AUDIT/'replay_test.csv',AUDIT/'all_access_audit.csv',EVAL/'paired_truth_evaluation.csv',REVIEW/'blinded_review_queue.csv']
for p in UPLOAD_ARTEFACTS: require(p.exists(),f'Missing upload artefact: {p}')
manifest=[]
for src in UPLOAD_ARTEFACTS:
    dst=UPLOAD/src.name; shutil.copy2(src,dst); manifest.append({'file':dst.name,'source':str(src),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'README.txt').write_text('Notebook 72 unified V8.8 operational trial review package. Start with executive_summary.txt, notebook72_verdict.json and prediction_lock.json. The full per-page image/probability/feature artefacts remain in the durable run directory.\n')
with zipfile.ZipFile(UPLOAD/'notebook72_upload_bundle.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in UPLOAD.iterdir():
        if p.is_file() and p.suffix!='.zip': z.write(p,p.name)
print('Upload bundle:',UPLOAD/'notebook72_upload_bundle.zip')

## Decision boundary

A successful run means the unified operator executed over the complete eligible reviewed roster, respected the truth firewall, reproduced its deterministic replay sample, preserved outside-mask bytes, and closed an auditable prediction lock before clean evaluation.

It does **not** by itself authorise automatic deployment. Review `blinded_review/blinded_review_queue.csv`, paired evaluation where available, source/population stability, failures/exclusions and the full per-page artefacts before any release decision.